Join the data from Part 1 with the data from Part 2 to create a new dataframe.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import sqlite3
conn = sqlite3.connect('mydatabase')



In [ ]:
pd.read_sql('select * from bike_stations b join yelp y on b.Station_Number = y.Nearest_Station join foursquare f on f.Nearest_Station = b.Station_Number', conn)

## Now if we want all our data we can join the 3 tables like this
## Because of the yelp api limitation we will end up with only 100 bike stations

In [ ]:
pd.read_sql('select * from ratings join bike_stations on ratings.Nearest_Station = bike_stations.Station_Number WHERE Station_Number == 5 order by ratings.Rating desc limit 10' , conn)
## We can check top rated spots near where we will park our bike




EDA and data cleaning

In [ ]:
fulldf=pd.read_sql('select * from bike_stations b join yelp y on b.Station_Number = y.Nearest_Station join foursquare f on f.Nearest_Station = b.Station_Number', conn)
fulldf = fulldf.drop(columns=['Nearest_Station'])  ## There was a duplicate from the join that we can remove


In [ ]:

print(fulldf.isnull().sum())         ## checking for nulls - everything looks good

In [ ]:
print( fulldf.duplicated().sum())  ## duplicate check also looks good

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


sns.histplot(fulldf['Foursquare_Results'])  ## a quick check of the distribution reveals no crazy outliers

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


sns.histplot(fulldf['Yelp_Results'])   ##  a quick check of the distribution reveals no crazy outliers

Since our data has no nulls , duplicates, or outliers, we can begin EDA

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


plt.scatter(x=fulldf['Longitude'],y=fulldf["Latitude"], c = fulldf['Yelp_Results'])

plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Location and Yelp Review Count')
plt.colorbar().set_label('Yelp Results')


## Here we can spot an initial pattern of the businesses clustering in certain areas.  We can use this as a hypothesis in our model- that location can predict the number of yelp results.


In [ ]:
fulldf.describe(include='all')
## not much noteworthy here, but 7.775 average free bikes per station might be useful

In [ ]:
correlation_matrix = fulldf.corr()
plt.figure(figsize=(10, 8))  # Adjust the figure size if needed
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=.5)
plt.title("Correlation Heatmap")
plt.show()

## Here we can see on first glance that free_bikes is probably not a good predictor of anything, but we will test that in the model building
## yelp and foursquare results are highly correlated which makes sense but isnt useful
## 